In [2]:
import sys, json
sys.path.append("..")
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import DataLoader
from src.tokenizer import CharTokenizer
from src.dataset import SQLDataset, collate
from src.model import GPTConfig, TinyGPT

def load(name):
    return [json.loads(line) for line in open(f"../data/raw/{name}.jsonl", encoding="utf-8")]

train = load("train")
tok = CharTokenizer.build([r["question"] for r in train] + [r["sql"] for r in train])
print("vocab size:", tok.vocab_size)

vocab size: 73


In [3]:
config = GPTConfig(vocab_size=tok.vocab_size)
model = TinyGPT(config)

loader = DataLoader(SQLDataset(train, tok), batch_size=4, shuffle=True,
                    collate_fn=lambda b: collate(b, tok.pad_id))
xb, yb = next(iter(loader))

out = model(xb)
print("input :", xb.shape)
print("output:", out.shape)

n_params = sum(p.numel() for p in model.parameters())
print("parameters:", n_params)

input : torch.Size([4, 253])
output: torch.Size([4, 253, 128])
parameters: 44160


In [4]:
from src.model import Head

config = GPTConfig(vocab_size=73)
head = Head(config, head_size=32)

# 1. Shapes
x = torch.randn(2, 10, 128)
print("output:", head(x).shape)                 # expect [2, 10, 32]

# 2. The causality test: change ONLY the last token, check who notices
head.eval()                                     # dropout off so results are repeatable
x = torch.randn(1, 8, 128)
out1 = head(x)
x2 = x.clone()
x2[:, -1, :] = torch.randn(128)                 # replace the last token with something else
out2 = head(x2)

print("earlier tokens unchanged:", torch.allclose(out1[:, :-1], out2[:, :-1]))   # expect True
print("last token changed:      ", not torch.allclose(out1[:, -1], out2[:, -1]))  # expect True

output: torch.Size([2, 10, 32])
earlier tokens unchanged: True
last token changed:       True
